# Week 4 · Hierarchical clustering, DBSCAN and Gaussian mixtures

k-means has three weaknesses (`03` §8): you must choose $k$, it only finds round blobs, and every point must belong somewhere. This notebook gives one method for each. Each comes with a small example you can do by hand, which is how the exam asks about them. The last section connects clustering to **anomaly detection**, one of the lab's research areas. **Reading:** ISL Ch. 12.4.2 (hierarchical); Bishop, *Pattern Recognition and Machine Learning*, Ch. 9.2 (mixtures and EM; the PDF is free from Microsoft Research).

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **linkage**: How the distance between two **clusters** is measured, given distances between points: closest pair (single), farthest pair (complete), or the average.
- **dendrogram**: The tree of merges. The height of each join is the distance at which the two clusters were merged.
- **$\varepsilon$ (eps), minPts**: DBSCAN's two settings: the neighbourhood radius, and how many points (including itself) a point needs within that radius to be a **core** point.
- **responsibility $r_{ik}$**: In a Gaussian mixture, the probability that point $i$ came from component $k$. A soft version of k-means' hard labels.
- **EM**: Expectation–maximization: alternate computing the responsibilities (E) and refitting the components with them (M).

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "04")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** Each method fixes one of k-means' weaknesses. §4 compares all of them on the same data; §5 uses two of them to find anomalies.

In [ ]:
fig_cluster_map()

| Section | What | Why it's here |
|---|---|---|
| E | start from the end | what each **finished** method stores, and what it does with a **new** point (one of them can't!) |
| 1 | hierarchical clustering | no $k$ up front: build the whole tree of merges, then cut it; the exam's single vs complete linkage drill |
| 2 | DBSCAN | clusters of any shape, and points that belong to no cluster (noise) |
| 3 | Gaussian mixtures and EM | soft assignments, stretched clusters, and a probability for every point; EM step by hand |
| 4 | which one when | all four methods on three datasets |
| 5 | clustering for anomaly detection | the lab's topic: "normal" is where the data is dense |
| 6 | build it yourself | 9 exercises: linkage, DBSCAN (core points, a new point, the whole algorithm), one EM iteration, then the EM loop |

## 0. Recap first · 10 min, closed book

**Quick check:** k-means: what are the two steps, and which quantity does each one lower?

<details><summary>Answer</summary>

Assign (each point to its nearest centroid) and update (each centroid to its points' mean). Both lower the inertia $J = \sum_i \lVert x_i - \mu_{c_i} \rVert^2$. [03 §5 ↗](03_kmeans.ipynb)

</details>

**Quick check:** Silhouette: $a = 1$, $b = 4$. What is $s$, and what does it say?

<details><summary>Answer</summary>

$\frac{4 - 1}{4} = 0.75$: the point sits well inside its own cluster. [03 §7 ↗](03_kmeans.ipynb)

</details>

**Quick check:** Bayes' rule: $P(A) = 0.5$, $P(x \mid A) = 0.2$, $P(B) = 0.5$, $P(x \mid B) = 0.05$, and $A$, $B$ are the only options. What is $P(A \mid x)$?

<details><summary>Answer</summary>

$\frac{0.5 \times 0.2}{0.5 \times 0.2 + 0.5 \times 0.05} = \frac{0.1}{0.125} = 0.8$. Section 3 uses exactly this. [Week 1 · math §1 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** The density of $N(\mu, \sigma^2)$ at $x$?

<details><summary>Answer</summary>

$\frac{1}{\sqrt{2\pi\sigma^2}}\, e^{-\frac{(x - \mu)^2}{2\sigma^2}}$. [Week 1 · math §3 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** Write `count_leaves(tree)` for the dictionary trees of Week 3 (a leaf is `{"leaf": 0}`; a node has `"left"` and `"right"`). It must work for any depth.

<details><summary>Answer</summary>

Base case: a leaf counts 1. Otherwise: the leaves on the left plus the leaves on the right. `return 1 if "leaf" in tree else count_leaves(tree["left"]) + count_leaves(tree["right"])`. Yes, it works for any depth: each call handles one node and trusts the calls below it, until every path reaches a leaf. (You couldn't write it on Nov 2; try it from memory now.) [00 §D ↗](00_week3_review.ipynb)

</details>

**Quick check:** Name the three weaknesses of k-means.

<details><summary>Answer</summary>

You must choose $k$; it assumes round, similar-sized blobs (on comparable scales); and every point must join a cluster, even outliers. [03 §8 ↗](03_kmeans.ipynb)

</details>

## E. Start from the end: three finished models, and a new point

As in [02 §E ↗](02_pca.ipynb) and [03 §E ↗](03_kmeans.ipynb): first what each **finished** method keeps, and what happens when a **new** point arrives. The answers differ more than you'd expect.

| Method | What's stored after fitting | A new point | scikit-learn |
|---|---|---|---|
| hierarchical | the tree of merges (who merged with whom, at what height) and a label per **training** point | **no rule**: the tree only describes the training points. Refit with the new point, or give it the cluster of its nearest training point | `AgglomerativeClustering` has **no** `predict` |
| DBSCAN | $\varepsilon$, which points are **core** points, and the cluster of each | joins the cluster of a core point within $\varepsilon$; otherwise it's **noise** | no `predict` either; do it by hand (exercise 7) |
| Gaussian mixture | $\pi_k$, $\mu_k$, $\sigma^2_k$ for every component | its responsibilities by Bayes' rule (soft cluster), the most likely component (hard cluster), and its density $p(x)$ (anomaly score) | `predict_proba`, `predict`, `score_samples` |

**DBSCAN: four new points**

The finished DBSCAN from §2 ($\varepsilon = 0.6$, minPts $= 3$): core points 1.5, 2 (cluster 0) and 8.5 (cluster 1). Border points (1, 2.5, 8, 9) belong to clusters, but **can't pass membership on**: only core points can.

| New point | Core points within 0.6 | Result |
|---|---|---|
| 1.9 | 1.5 (0.4 away), 2 (0.1 away) | cluster 0 |
| 2.8 | none: 2 is 0.8 away; 2.5 is only 0.3 away, but it's a border point | **noise** |
| 8.9 | 8.5 (0.4 away) | cluster 1 |
| 6.5 | none | noise |

**Gaussian mixture: a new point**

Take the mixture after one EM step from §3: $\pi = (0.408, 0.592)$, $\mu = (2.324, 6.0)$, $\sigma^2 = (2.536, 4.424)$. For $x = 4.5$: $\pi_1 \mathcal{N}(4.5 \mid 2.324, 2.536) = 0.0402$ and $\pi_2 \mathcal{N}(4.5 \mid 6, 4.424) = 0.0871$, so the responsibilities are $\frac{0.0402}{0.1273} = 0.316$ and $0.684$, and the density is $p(4.5) = 0.1273$. For $x = 10$: $r = (0.000, 1.000)$ but $p(10) = 0.0184$, seven times less likely, which is how a GMM flags an unusual point (§5).

🖼 **Picture.** Top: the finished DBSCAN and the four new points. Bottom: the finished mixture; each new point gets a probability for each bump and a density.

In [ ]:
fig_new_points_clusterings()

In [ ]:
x_train = np.array([1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12.0])
labels = np.array([0, 0, 0, 0, -1, 1, 1, 1, -1])                  # what DBSCAN stored (§2)
core = np.array([False, True, True, False, False, False, True, False, False])
for x_new in [1.9, 2.8, 8.9, 6.5]:
    near = core & (np.abs(x_train - x_new) <= 0.6)                 # CORE points within eps
    print(x_new, labels[near][0] if near.any() else -1)
# → 1.9 0
#   2.8 -1
#   8.9 1
#   6.5 -1

In [ ]:
pdf = lambda x, m, v: np.exp(-(x - m) ** 2 / (2 * v)) / np.sqrt(2 * np.pi * v)
pi, mu, var = np.array([0.408, 0.592]), np.array([2.324, 6.0]), np.array([2.536, 4.424])
for x_new in [4.5, 10.0]:
    joint = pi * pdf(x_new, mu, var)                               # (2,): prior × likelihood for each bump
    print(x_new, (joint / joint.sum()).round(3), round(joint.sum(), 4))
# → 4.5 [0.316 0.684] 0.1273
#   10.0 [0. 1.] 0.0184

**The rest of the notebook answers: where do these stored things come from?** The merge tree is §1, the core points §2, the bumps §3 (EM). Keep the end result in mind and each training step has an obvious purpose.

## 1. Hierarchical clustering

**Agglomerative clustering starts with every point as its own cluster and repeatedly merges the two closest clusters, until only one is left. The record of merges is a tree; cutting it at some height gives any number of clusters you like.**

**"Closest clusters" needs a rule, the linkage.** With $d(a, b)$ the distance between two points:

| Linkage | Distance between clusters $A$ and $B$ | Tends to give |
|---|---|---|
| single | $\min_{a \in A,\ b \in B} d(a, b)$: the closest pair | long chains (one close pair is enough to merge) |
| complete | $\max_{a \in A,\ b \in B} d(a, b)$: the farthest pair | compact, similar-sized clusters |
| average | the mean of all $d(a, b)$ | something in between |
| Ward (scikit-learn's default) | the increase in within-cluster squared error caused by the merge | k-means-like round clusters |

**The exam drill: five points on a line**, $1, 2, 4, 8, 13$. First, every pairwise distance:

|  | 1 | 2 | 4 | 8 | 13 |
|---|---|---|---|---|---|
| 1 | 0 | 1 | 3 | 7 | 12 |
| 2 | 1 | 0 | 2 | 6 | 11 |
| 4 | 3 | 2 | 0 | 4 | 9 |
| 8 | 7 | 6 | 4 | 0 | 5 |
| 13 | 12 | 11 | 9 | 5 | 0 |

**Single linkage, merge by merge**

- Smallest distance: $d(1, 2) = 1$. Merge them into $\{1, 2\}$ at height **1**.
- Distances now: $\{1, 2\}$ to $4$ is $\min(3, 2) = 2$; $4$ to $8$ is 4; $8$ to $13$ is 5. Merge $\{1, 2, 4\}$ at height **2**.
- $\{1, 2, 4\}$ to $8$ is $\min(7, 6, 4) = 4$; $8$ to $13$ is 5. Merge $\{1, 2, 4, 8\}$ at height **4**.
- Finally $\{1, 2, 4, 8\}$ to $13$ is $\min(12, 11, 9, 5) = 5$. Merge everything at height **5**.

**Complete linkage, merge by merge**

- $\{1, 2\}$ at height **1**, as before.
- $\{1, 2\}$ to $4$ is now $\max(3, 2) = 3$; $4$ to $8$ is 4; $8$ to $13$ is 5. Merge $\{1, 2, 4\}$ at height **3**.
- $\{1, 2, 4\}$ to $8$ is $\max(7, 6, 4) = 7$, but $8$ to $13$ is only 5. Merge $\{8, 13\}$ at height **5**.
- Finally $\{1, 2, 4\}$ to $\{8, 13\}$ is $\max(\dots) = 13 - 1 = 12$. Merge everything at height **12**.

🖼 **Picture.** The two trees. The height of each horizontal bar is the merge distance from the steps above.

In [ ]:
fig_dendrograms()

**Cutting for $k = 2$** (one level below the top): single linkage gives $\{1, 2, 4, 8\}$ and $\{13\}$; complete linkage gives $\{1, 2, 4\}$ and $\{8, 13\}$. Single linkage **chains**: 8 joined the left group because its nearest neighbour (4) was there, even though 8 is far from 1 and 2. Complete linkage asks that **all** members be close.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
pts = np.array([1, 2, 4, 8, 13.0])[:, None]                 # (5, 1): 5 points, 1 feature
for how in ["single", "complete"]:
    Z = linkage(pts, how)                                     # one row per merge: [cluster a, cluster b, height, size]
    print(how, Z[:, 2].tolist(), fcluster(Z, 2, criterion="maxclust").tolist())
# → single [1.0, 2.0, 4.0, 5.0] [1, 1, 1, 1, 2]
#   complete [1.0, 3.0, 5.0, 12.0] [1, 1, 1, 2, 2]

**Properties.** No $k$ needed in advance; the dendrogram shows every possible $k$ at once. Cost: it needs all $\frac{n(n - 1)}{2}$ pairwise distances, so it's slow beyond roughly $10^4$ points. Merges are never undone (greedy, like trees).

## 2. DBSCAN: clusters are dense regions

**DBSCAN grows clusters from points that have many neighbours, and leaves isolated points out as noise. It finds clusters of any shape, and you don't choose their number.**

| Kind of point | Rule |
|---|---|
| **core** | at least minPts points (itself included) within distance $\varepsilon$ |
| **border** | not core, but within $\varepsilon$ of some core point |
| **noise** | neither: belongs to no cluster |

**Clusters**: two core points within $\varepsilon$ of each other are in the same cluster, and so is everything connected to them through such chains. Border points join the cluster of a core point near them.

**Worked example (1-D):** $x = 1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12$ with $\varepsilon = 0.6$, minPts $= 3$.

| $x$ | neighbours within 0.6 (incl. itself) | count | kind | cluster |
|---|---|---|---|---|
| 1 | 1, 1.5 | 2 | border (near core 1.5) | 0 |
| 1.5 | 1, 1.5, 2 | 3 | **core** | 0 |
| 2 | 1.5, 2, 2.5 | 3 | **core** | 0 |
| 2.5 | 2, 2.5 | 2 | border (near core 2) | 0 |
| 5 | 5 | 1 | noise | – |
| 8 | 8, 8.5 | 2 | border | 1 |
| 8.5 | 8, 8.5, 9 | 3 | **core** | 1 |
| 9 | 8.5, 9 | 2 | border | 1 |
| 12 | 12 | 1 | noise | – |

🖼 **Picture.** The same table, drawn. Orange = core, light orange = border, grey = noise. The shaded windows show each core point's neighbourhood.

In [ ]:
fig_dbscan_1d()

In [ ]:
from sklearn.cluster import DBSCAN
x = np.array([1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12.0])[:, None]    # (9, 1)
db = DBSCAN(eps=0.6, min_samples=3).fit(x)
print(db.labels_)                        # → [ 0  0  0  0 -1  1  1  1 -1]   -1 = noise
print(db.core_sample_indices_)           # → [1 2 6]   the core points: 1.5, 2, 8.5

🖼 **Picture.** Two moons plus 20 random noise points. k-means cuts straight across both moons and forces the noise into clusters; DBSCAN follows the curves and labels the noise (black).

In [ ]:
fig_dbscan_vs_kmeans()

**Choosing $\varepsilon$.** For every point, find the distance to its (minPts − 1)-th nearest neighbour, sort, and plot. Points inside clusters have small values; noise has large ones. Put $\varepsilon$ just below the bend:

🖼 **Picture.** The sorted 4th-neighbour distances for the moons data: flat around 0.1, then a sharp bend at the noise points. $\varepsilon = 0.2$ sits just below it.

In [ ]:
fig_kdistance()

**Properties.** Any shape; finds noise; no $k$. But one $\varepsilon$ for the whole dataset, so it struggles when clusters have very **different densities**; and distances need scaled features, like kNN.

## 3. Gaussian mixtures and the EM algorithm

**A Gaussian mixture model (GMM) says each point was produced by first picking one of $K$ Gaussian bumps at random (bump $k$ with probability $\pi_k$), then drawing the point from that bump. Fitting it gives every point a probability of belonging to each cluster.**

$$
p(x) = \sum_{k=1}^{K} \pi_k\, \mathcal{N}(x \mid \mu_k, \sigma_k^2), \qquad \pi_k \ge 0,\ \ \sum_k \pi_k = 1
$$

$\pi_k$ is the mixing weight (how common cluster $k$ is), and $\mu_k$, $\sigma_k^2$ are bump $k$'s centre and spread. In more than one dimension, each bump has a mean vector and a covariance matrix, so it can be a tilted ellipse.

**The responsibility.** Given a point $x_i$, which bump made it? Bayes' rule, with prior $\pi_k$ and likelihood $\mathcal{N}(x_i \mid \mu_k, \sigma_k^2)$:

$$
r_{ik} = P(\text{bump } k \mid x_i) = \frac{\pi_k\, \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)}{\sum_{j} \pi_j\, \mathcal{N}(x_i \mid \mu_j, \sigma_j^2)}
$$

**EM fits the model by alternating two steps**, like k-means' assign and update, but soft:

| Step | k-means | EM for a GMM |
|---|---|---|
| E (expectation) | assign each point to ONE nearest centroid | compute every $r_{ik}$: a share of each point for every bump |
| M (maximization) | centroid = mean of its points | $N_k = \sum_i r_{ik}$, $\ \mu_k = \frac{1}{N_k}\sum_i r_{ik} x_i$, $\ \sigma_k^2 = \frac{1}{N_k}\sum_i r_{ik}(x_i - \mu_k)^2$, $\ \pi_k = \frac{N_k}{n}$ |

The M-step is an ordinary mean and variance, except each point counts with its responsibility as a weight: a point that's 87% bump 1 counts 0.87 toward bump 1's mean.

**One EM iteration by hand**

Points $x = 1, 2, 3, 6, 7, 8$. Start: $\mu = (2, 5)$, $\sigma^2 = (4, 4)$, $\pi = (0.5, 0.5)$. The bump at 5 starts in the wrong place on purpose.

🖼 **Picture.** Top: the two starting bumps (weighted by $\pi$) and their sum. Bottom: each point's responsibility for bump 2.

In [ ]:
fig_gmm_1d()

| $x$ | $\pi_1 \mathcal{N}(x \mid 2, 4)$ | $\pi_2 \mathcal{N}(x \mid 5, 4)$ | $r_{i1}$ | $r_{i2}$ |
|---|---|---|---|---|
| 1 | 0.0880 | 0.0135 | 0.867 | 0.133 |
| 2 | 0.0997 | 0.0324 | 0.755 | 0.245 |
| 3 | 0.0880 | 0.0605 | 0.593 | 0.407 |
| 6 | 0.0135 | 0.0880 | 0.133 | 0.867 |
| 7 | 0.0044 | 0.0605 | 0.068 | 0.932 |
| 8 | 0.0011 | 0.0324 | 0.033 | 0.967 |

Row $x = 1$ by hand: $\mathcal{N}(1 \mid 2, 4) = \frac{1}{\sqrt{8\pi}} e^{-1/8} = 0.1995 \times 0.8825 = 0.1760$, times $\pi_1 = 0.5$ gives 0.0880. Likewise $0.5\,\mathcal{N}(1 \mid 5, 4) = 0.5 \times 0.1995 \times e^{-2} = 0.0135$. So $r_{11} = \frac{0.0880}{0.0880 + 0.0135} = 0.867$.

- $N_1 = 0.867 + 0.755 + 0.593 + 0.133 + 0.068 + 0.033 = 2.448$, and $N_2 = 6 - 2.448 = 3.552$.
- $\mu_1 = \frac{0.867 \cdot 1 + 0.755 \cdot 2 + 0.593 \cdot 3 + 0.133 \cdot 6 + 0.068 \cdot 7 + 0.033 \cdot 8}{2.448} = 2.324$; likewise $\mu_2 = 6.000$.
- $\sigma_1^2 = 2.536$ and $\sigma_2^2 = 4.424$ (weighted variances around the new means).
- $\pi = \big(\frac{2.448}{6}, \frac{3.552}{6}\big) = (0.408, 0.592)$.
- The log-likelihood $\sum_i \ln p(x_i)$ rose from $-14.638$ to $-13.729$. **EM never lowers it**, the way k-means never raises $J$.

🖼 **Picture.** After one iteration. The responsibilities are already sharper: the points at 6, 7, 8 are now almost entirely bump 2's.

In [ ]:
fig_gmm_1d(mu=(2.324, 6.0), var=(2.536, 4.424), pi=(0.408, 0.592), title="After one EM step: bump 2 moved from 5 to 6, onto its points")

In [ ]:
x = np.array([1, 2, 3, 6, 7, 8.0])
pdf = lambda x, m, v: np.exp(-(x - m) ** 2 / (2 * v)) / np.sqrt(2 * np.pi * v)
mu, var, pi = np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5])
N = np.stack([pi[k] * pdf(x, mu[k], var[k]) for k in range(2)], axis=1)    # (6, 2)
R = N / N.sum(axis=1, keepdims=True)                                         # E-step: each row sums to 1
Nk = R.sum(axis=0)
mu = (R * x[:, None]).sum(axis=0) / Nk                                       # M-step
var = (R * (x[:, None] - mu) ** 2).sum(axis=0) / Nk
print(R[:, 0].round(3), Nk.round(3), mu.round(3), var.round(3), (Nk / 6).round(3))
# → [0.867 0.755 0.593 0.133 0.068 0.033] [2.448 3.552] [2.324 6.   ] [2.536 4.424] [0.408 0.592]

🖼 **Picture.** EM in 2-D with 3 components and full covariance matrices (tilted ellipses). After 30 steps the ellipses fit the three groups, including the stretched one, which k-means can't describe.

In [ ]:
fig_em_iterations()

**k-means is a special case:** make every bump round with the same tiny variance, and each responsibility becomes 0 or 1 (all of a point goes to the nearest centre), which is k-means' assign step. **Choosing $K$:** compare the BIC (`GaussianMixture(...).bic(X)`), which rewards a high likelihood but charges for extra parameters; lower is better.

## 4. Which one when

🖼 **Picture.** Four methods, three datasets (all standardized). Black = DBSCAN noise. Only DBSCAN follows the moons; only the Gaussian mixture handles the stretched blob; Ward and k-means agree on round blobs.

In [ ]:
fig_compare_clustering()

|  | k-means | hierarchical | DBSCAN | Gaussian mixture |
|---|---|---|---|---|
| you choose | $k$ | where to cut (or $k$) | $\varepsilon$, minPts | $K$, covariance type |
| cluster shapes | round | depends on linkage | any | ellipses |
| assignments | hard | hard | hard, plus noise | soft (probabilities) |
| outliers | forced into a cluster | forced (or end up alone) | labelled noise | low density, so detectable |
| size limit | millions | ~$10^4$ (all pairwise distances) | large, with an index | large |

## 5. Clustering for anomaly detection

**If clustering describes where normal data lives, an anomaly is a point that fits no cluster.** Two of the methods give an anomaly score for free:

- **DBSCAN**: points labelled noise (−1) are anomalies.
- **Gaussian mixture**: fit it to normal data, then score new points by their log-density $\ln p(x)$. A very low density means "the model has rarely seen anything like this". Flag points below a threshold, e.g. the 1st percentile of the training scores.

🖼 **Picture.** A GMM fitted to normal data; grey contours are levels of $\ln p(x)$. Points outside the lowest contour (red ✗) are flagged.

In [ ]:
fig_anomaly_gmm()

This is the idea behind the lab's **anomaly detection** work and your **KAMP** alerts: learn what normal operation looks like, then flag what doesn't fit. The deep-learning version (Week 7) replaces the Gaussian bumps with an autoencoder's reconstruction error, but the logic is the same: **normal = dense, anomaly = unlikely under the model of normal**. Picking the threshold is then a precision–recall trade-off ([Week 3 · 01 §9 ↗](../week3/01_logistic_regression_in_pictures.ipynb)).

## 6. Build it yourself

✏️ **Exercise 1 · Linkage distance.** `A` is `(m, d)` and `B` is `(p, d)`: two clusters. Return the single (min), complete (max) or average distance between them, depending on `how`. Build the `(m, p)` table of all pairwise distances first.

**Example**: what goes in, and what should come out:
```python
A = np.array([[1.0], [2.0]]); B = np.array([[4.0]])     # clusters {1, 2} and {4}, 1-D
linkage_distance(A, B, "single")      # → 2.0
linkage_distance(A, B, "complete")    # → 3.0
linkage_distance(A, B, "average")     # → 2.5
```

In [ ]:
def linkage_distance(A, B, how):
    pass  # your code

In [ ]:
check(1)

✏️ **Exercise 2 · DBSCAN core points.** Return a True/False array, **shape `(n,)`**: True where a point has at least `min_pts` points (itself included) within distance `eps`. One `(n, n)` distance table, then a comparison and a sum along one axis.

**Example**: what goes in, and what should come out:
```python
X = np.array([1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12.0])[:, None]     # (9, 1)
dbscan_core(X, 0.6, 3)
# → [False  True  True False False False  True False False]
```

In [ ]:
def dbscan_core(X, eps, min_pts):
    pass  # your code

In [ ]:
check(2)

✏️ **Exercise 3 · Gaussian density.** $\mathcal{N}(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\, e^{-\frac{(x - \mu)^2}{2\sigma^2}}$, for a number or an array `x` (element by element).

**Example**: what goes in, and what should come out:
```python
gaussian_pdf(1.0, 2.0, 4.0)                         # → 0.1760
gaussian_pdf(np.array([1.0, 2.0]), 2.0, 4.0)        # → [0.176  0.1995]
```

In [ ]:
def gaussian_pdf(x, mu, var):
    pass  # your code

In [ ]:
check(3)

✏️ **Exercise 4 · E-step.** `x`: `(n,)`; `mu`, `var`, `pi`: `(K,)`. Return the responsibilities, **shape `(n, K)`**: column `k` is $\pi_k \mathcal{N}(x \mid \mu_k, \sigma_k^2)$, then each row divided by its sum (`keepdims=True`).

**Example**: what goes in, and what should come out:
```python
x = np.array([1, 2, 3, 6, 7, 8.0])
R = e_step(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]))
R[:, 0]     # → [0.867 0.755 0.593 0.133 0.068 0.033]     R.shape = (6, 2), rows sum to 1
```

In [ ]:
def e_step(x, mu, var, pi):
    pass  # your code

In [ ]:
check(4)

✏️ **Exercise 5 · M-step.** Return `(mu, var, pi)`, each **shape `(K,)`**, from the formulas in §3: $N_k$, the weighted means, the weighted variances around the **new** means, and $N_k / n$.

**Example**: what goes in, and what should come out:
```python
mu, var, pi = m_step(x, R)     # with x and R from exercise 4
mu     # → [2.324 6.   ]
var    # → [2.536 4.424]
pi     # → [0.408 0.592]
```

In [ ]:
def m_step(x, R):
    pass  # your code

In [ ]:
check(5)

✏️ **Exercise 6 · Log-likelihood.** $\sum_i \ln\Big(\sum_k \pi_k \mathcal{N}(x_i \mid \mu_k, \sigma_k^2)\Big)$. **Shape `()`.** Run E and M a few times and watch it rise.

**Example**: what goes in, and what should come out:
```python
log_likelihood(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]))   # → -14.638
log_likelihood(x, np.array([2.324, 6.0]), np.array([2.536, 4.424]), np.array([0.408, 0.592]))   # → -13.729
```

In [ ]:
def log_likelihood(x, mu, var, pi):
    pass  # your code

In [ ]:
check(6)

**Now the methods as complete programs**: what section E said each one keeps, and how it's used.

✏️ **Exercise 7 · DBSCAN: a new point.** Given what a finished DBSCAN stored (`X_train` `(n, d)`, `labels` `(n,)`, `core` True/False `(n,)`) and a new point `x_new` `(d,)`, return the label of the **nearest core point** within `eps`, or `-1` if there is none.

**Example**: what goes in, and what should come out:
```python
X_train = np.array([1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12.0])[:, None]      # (9, 1)
labels = np.array([0, 0, 0, 0, -1, 1, 1, 1, -1])
core = np.array([False, True, True, False, False, False, True, False, False])
dbscan_predict(X_train, labels, core, np.array([1.9]), 0.6)    # → 0
dbscan_predict(X_train, labels, core, np.array([2.8]), 0.6)    # → -1   (2.5 is near, but not core)
```

In [ ]:
def dbscan_predict(X_train, labels, core, x_new, eps):
    pass  # your code

In [ ]:
check(7)

✏️ **Exercise 8 · DBSCAN from scratch.** Return the labels, `(n,)`: clusters numbered 0, 1, ... and noise $-1$. **Recipe:** (1) find the core points (your exercise 2). (2) Go through the points in order; when you meet a core point with no label yet, start a new cluster: put it in a to-do list, and while the list isn't empty, take a point out, and give every unlabelled neighbour within `eps` the cluster number; neighbours that are **core** points also go into the to-do list. (3) Everything left unlabelled is noise. The checker compares the grouping with scikit-learn's `DBSCAN` (the cluster numbers may differ).

**Example**: what goes in, and what should come out:
```python
X = np.array([1, 1.5, 2, 2.5, 5, 8, 8.5, 9, 12.0])[:, None]
dbscan(X, 0.6, 3)     # → [ 0  0  0  0 -1  1  1  1 -1]
```

In [ ]:
def dbscan(X, eps, min_pts):
    pass  # your code

In [ ]:
check(8)

✏️ **Exercise 9 · EM, the whole loop.** Run `iters` EM iterations from the given starting values using your `e_step` and `m_step`, and return the final `(mu, var, pi)`. After that, `e_step(x_new, mu, var, pi)` **is** the finished model's `predict_proba`.

**Example**: what goes in, and what should come out:
```python
x = np.array([1, 2, 3, 6, 7, 8.0])
mu, var, pi = gmm_fit_1d(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]), 1)
mu     # → [2.324 6.   ]     (one iteration: the numbers from §3)
mu, var, pi = gmm_fit_1d(x, np.array([2.0, 5.0]), np.array([4.0, 4.0]), np.array([0.5, 0.5]), 50)
mu     # → about [2. 7.]     the two groups
```

In [ ]:
def gmm_fit_1d(x, mu, var, pi, iters):
    pass  # your code

In [ ]:
check(9)

In [ ]:
for n in range(1, 10):
    check(n)

---
Solutions: `solutions/04_solutions.py`. Next: `05_ml_project.ipynb`.